---
title: "DRG Cleaning"

author: "Carlos Resurreccion"

date: "2024-07-01"

---

## Parameters

In [19]:
# IMPORTANT PARAMETERS:
year_to_load <- "2018"
version <- "v2"
split_chunks <- 5
rows_to_show <- 10

# Input:
to_read <- FALSE
to_split <- TRUE
to_sample <- TRUE
to_loop <- TRUE # doesn't work if false; not looping is deprecated.

# Output:
to_write <- TRUE
to_group <- TRUE

# Debug:
to_profvis <- FALSE
to_chunk <- TRUE # doesn't work if false; not chunking is deprecated.
to_view_checks <- TRUE
to_view_checks_parallelized <- FALSE
to_parallelize <- TRUE

# Sample size divisor:
sample_size_divisor <- 5

if (to_loop) {
  split_chunk_to_process <- NA
} else {
  stop("not looping is deprecated.")
  # Function to safely read a numeric input within a specific range
  read_number_in_range <- function(prompt, min_value, max_value) {
    repeat {
      input <- readline(prompt = prompt)
      num <- suppressWarnings(as.numeric(input))
      if (!is.na(num) && num >= min_value && num <= max_value) {
        return(num)
      }
      cat(
        "Invalid input. Please enter a number between",
        min_value, "and", max_value, ".\n"
      )
    }
  }

  # Prompt the user for a number within the range
  split_chunk_to_process <- read_number_in_range(
    paste("Enter a number between 1 and", split_chunks, ":"), 1, split_chunks
  )

  # Display the number
  cat("You entered the number:", split_chunk_to_process, "\n")

  # Use the number in your code
  # For example, processing the specified chunk
  cat("Processing chunk number", split_chunk_to_process, "...\n")

  # Example of using the number in a loop or conditional
  if (split_chunk_to_process == 1) {
    cat("You selected the first chunk.\n")
  } else {
    cat("You selected chunk number", split_chunk_to_process, ".\n")
  }
}

if (is.na(split_chunk_to_process)) {
  part <- NULL
} else {
  part <- split_chunk_to_process
}

drop_cols <- c(
  paste0("ICDCODE", 13:14),
  "ICCODED15",
  paste0("ICDCODE", 16:170)
)

icd_cols <- paste0("clin_icd", 1:12)
rvs_cols <- paste0("clin_rvs", 1:20)

seed <- 123
set.seed(seed)

options(future.globals.maxSize = 1024 * 1024^2)

global_seed <- seed # for parallelized operations

## Load Required Libraries & Initial Functions

In [20]:
options(verbose = FALSE)
options(warn = -1)
library(here)


In [21]:
# List of scripts to source in order
scripts_to_source <- c(
  "00_libraries.R",
  "01_data-formats.R",
  "02_file-paths.R"
)

# Generate and execute source commands
for (script in scripts_to_source) {
  source(here("data-cleaning", "r_scripts", script))
}

# Perform other tasks here...
tic("Total execution time:")

# Use the function to count total rows
if (file.exists(full_claims_file(part))) {
  total_rows <- fread(full_claims_file(part), select = 1L, header = TRUE)[, .N]
} else {
  total_rows <- fread(full_claims_file(), select = 1L, header = TRUE)[, .N]
}
print(paste("Total Rows via fread:", total_rows))

sample_size_divisor <- 25

if (to_split) {
  if (is.null(part)) {
    sample_size <- ceiling(total_rows / split_chunks / sample_size_divisor)
  } else {
    sample_size <- ceiling(total_rows / split_chunks / sample_size_divisor)
  }
} else {
  if (is.null(part)) {
    sample_size <- ceiling(total_rows / sample_size_divisor)
  } else {
    sample_size <- ceiling(total_rows / sample_size_divisor)
  }
}

# List of scripts to source in order
scripts_to_source <- c(
  "03_general-functions.R",
  "04_main-functions.R",
  "05_io-functions.R",
  "06_icd-functions.R",
  "07_rvs-functions.R",
  "08_pdx-functions.R",
  "09_grouper-functions.R",
  "10_timing-functions.R",
  "11_debug-functions.R"
)

# Generate and execute source commands
for (script in scripts_to_source) {
  source(here("data-cleaning", "r_scripts", script))
}


[1] "Total Rows via fread: 11777674"


In [22]:
options(warn = 1)


## Load Mapping Data

In [23]:
proc <- fread(here(path_to_excel, "proc.csv"))
proc[, CODE := as.character(CODE)]
# head(proc)

rvs_icd9 <- fread(here(path_to_aux, "rvs_icd9cm.csv"),
  select = c("rvs", "icd9cm")
)
rvs_icd9[, rvs := as.character(rvs)]
rvs_icd9[, icd9cm := as.character(icd9cm * 100)]
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)],
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
)
rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE]
rvs_icd9 <- rvs_icd9[!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]

acr_rvs <- fread(here(path_to_aux, "acr_rvs.csv"))

# Read in the data.table
tdrg_icd10 <- fread(here(path_to_aux, "i10.csv"))

# Set the key if not already set
setkey(tdrg_icd10, "CODE")

# Subset and assign the result to acc_pdx
acc_pdx <- tdrg_icd10[ACCPDX == "Y", CODE]

# Optional: if CODEs are not unique in tdrg_icd10
acc_pdx <- unique(acc_pdx)


## Read, Process, Export Data (Looping through all parts)

In [24]:
all_parts_summaries <- list()
all_parts_statistics <- list()

if (!is.na(split_chunk_to_process)) {
  split_and_save_chunks(split_chunk_to_process)
  if (to_profvis) {
    p <- profvis({
      dt <- read_and_process_chunk(split_chunk_to_process)
      if (!is.null(dt) && nrow(dt) > 0) {
        dt <- parallelize_and_summarize_data(split_chunk_to_process, dt)
        write_intermediate_file(split_chunk_to_process, dt)
        group_data(split_chunk_to_process, dt)
      } else {
        print(paste("No data to process for part:", split_chunk_to_process))
      }
      combine_and_print_summaries()
    })
    htmlwidgets::saveWidget(
      p,
      file = here(
        "git-ignored-files", "profvis",
        paste0("profvis_part_", split_chunk_to_process, "_.html")
      ),
      selfcontained = TRUE
    )
  } else {
    dt <- read_and_process_chunk(split_chunk_to_process)
    if (!is.null(dt) && nrow(dt) > 0) {
      dt <- parallelize_and_summarize_data(split_chunk_to_process, dt)
      write_intermediate_file(split_chunk_to_process, dt)
      group_data(split_chunk_to_process, dt)
    }
    combine_and_print_summaries()
  }
} else {
  split_and_save_chunks()
  if (to_profvis) {
    p <- profvis({
      for (part in 1:split_chunks) {
        dt <- read_and_process_chunk(part)
        if (!is.null(dt) && nrow(dt) > 0) {
          dt <- parallelize_and_summarize_data(part, dt)
          write_intermediate_file(part, dt)
          group_data(part, dt)
        } else {
          print(paste("No data to process for part:", part))
        }
      }
      combine_and_print_summaries()
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "profvis.html"),
      selfcontained = TRUE
    )
  } else {
    for (part in 1:split_chunks) {
      dt <- read_and_process_chunk(part)
      if (!is.null(dt) && nrow(dt) > 0) {
        dt <- parallelize_and_summarize_data(part, dt)
        write_intermediate_file(part, dt)
        group_data(part, dt)
      } else {
        print(paste("No data to process for part:", part))
      }
    }
    combine_and_print_summaries()
  }
}

[1] "Checking full file: C:/Users/resur/Documents/drg-pipeline/git-ignored-files/raw-claims/samples/sampled_claims_2018_94222_part_1_of_5.csv"
[1] "Saving partial file: C:/Users/resur/Documents/drg-pipeline/git-ignored-files/raw-claims/parts/full_claims_2018_part_1_of_5.csv"
[1] "Creating sampled file: C:/Users/resur/Documents/drg-pipeline/git-ignored-files/raw-claims/samples/sampled_claims_2018_94222_part_1_of_5.csv"
[1] "Sampling from 2355535 rows to 94222 rows"
[1] "Sampled rows: 94222"
[1] "Sampled and saved file: C:/Users/resur/Documents/drg-pipeline/git-ignored-files/raw-claims/samples/sampled_claims_2018_94222_part_1_of_5.csv with rows: 94222"
[1] "Checking full file: C:/Users/resur/Documents/drg-pipeline/git-ignored-files/raw-claims/samples/sampled_claims_2018_94222_part_2_of_5.csv"
[1] "Saving partial file: C:/Users/resur/Documents/drg-pipeline/git-ignored-files/raw-claims/parts/full_claims_2018_part_2_of_5.csv"
[1] "Creating sampled file: C:/Users/resur/Documents/drg-pipeline

## Runtime Estimation

### Stop Timer

In [25]:
# Stop the timer and capture total time
toc_data <- toc(log = TRUE)
total_time <- toc_data$toc - toc_data$tic


Total execution time:: 253.02 sec elapsed


### Calculate Speed

In [26]:
if (!is.na(split_chunk_to_process)) {
  if (to_sample) {
    total_rows <- nrow(dt) * split_chunks * sample_size_divisor
  } else {
    total_rows <- nrow(dt) * split_chunks
  }
  print_time_estimates(split_chunk_to_process, dt, total_time, total_rows)
} else {
  if (to_sample) {
    total_rows <- nrow(dt) * split_chunks * sample_size_divisor
  } else {
    total_rows <- nrow(dt) * split_chunks
  }
  print_time_estimates(NA, dt, total_time, total_rows)
}


Time spent (total) for 11.8m rows:  253.02 sec  (actual)
Time spent (t/row) for 11.8m rows:  2.69 msec (actual)


## Debugging

In [27]:
library(here)
source(here("data-cleaning", "r_scripts", "11_debug-functions.R"))

input_path <- here("data-cleaning", "r_scripts")
output_file <- paste0(here("data-cleaning", "everything", "everything.R"))

concatenate_r_files(input_path, output_file)


To extract all code portions of this ipynb file (run in VS Code terminal):

jupyter nbconvert --no-prompt --to script data-cleaning/drg-cleaning.ipynb